In [ ]:
!pip install pandas openpyxl
# =====================================================
# Step-5 Implement Sale and Return logic
#identity + brand + model + billing_value
# =====================================================

import pandas as pd
from google.colab import files

# =====================================================
# UPLOAD FILE
# =====================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name, dtype=str)

# =====================================================
# ✅ FIX: CLEAN COLUMN NAMES
# =====================================================
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace("event_props.", "", regex=False)

# =====================================================
# NORMALIZE INVOICE TYPE (SALE / RETURN)
# =====================================================
def classify(row):
    t = str(row.get("Invoice Type", "")).lower()

    if "return" in t or "refund" in t:
        return "return"

    try:
        val = row.get("Total Billing Value", "")
        if float(str(val).replace(",", "")) < 0:
            return "return"
    except:
        pass

    return "sale"

df["inv_type_norm"] = df.apply(classify, axis=1)

# =====================================================
# DETECT ITEM COLUMNS (DYNAMIC)
# =====================================================
item_nums = sorted(set([
    int(col.split("_")[0].replace("Item",""))
    for col in df.columns
    if col.startswith("Item") and "Invoice Brand" in col
]))

# =====================================================
# EXPLODE MULTI ITEMS
# =====================================================
items = []

for _, row in df.iterrows():
    for i in item_nums:

        brand = row.get(f"Item{i}_Invoice Brand", "")
        model = row.get(f"Item{i}_Model Number", "")
        billing = row.get(f"Item{i}_Billing Value", "")

        if pd.notna(brand) and str(brand).strip() != "":
            try:
                billing_abs = abs(float(str(billing).replace(",", "")))
            except:
                billing_abs = None

            items.append({
                "orig_index": row.name,
                "identity": row.get("identity", ""),
                "brand": str(brand).strip(),
                "model": str(model).strip(),
                "billing_abs": billing_abs,
                "inv_type": row["inv_type_norm"],
                "ts": row.get("ts", "")
            })

items_df = pd.DataFrame(items)

# =====================================================
# MATCH SALE vs RETURN
# =====================================================
items_df["match_key"] = (
    items_df["identity"].astype(str) + "||" +
    items_df["brand"].astype(str) + "||" +
    items_df["model"].astype(str) + "||" +
    items_df["billing_abs"].astype(str)
)

to_remove = set()

for key, grp in items_df.groupby("match_key"):

    sales = grp[grp["inv_type"] == "sale"].sort_values("ts")
    returns = grp[grp["inv_type"] == "return"].sort_values("ts")

    n = min(len(sales), len(returns))

    if n > 0:
        to_remove.update(sales.iloc[:n]["orig_index"])
        to_remove.update(returns.iloc[:n]["orig_index"])

# =====================================================
# FINAL CLEAN DATA
# =====================================================
clean_df = df.drop(index=list(to_remove)).reset_index(drop=True)
clean_df = clean_df.drop(columns=["inv_type_norm"], errors="ignore")

# =====================================================
# SAVE OUTPUT
# =====================================================
out_name = "cleaned_no_sale_return_multiitem.xlsx"
clean_df.to_excel(out_name, index=False)

files.download(out_name)

print(f"✅ Done | Removed Rows: {len(to_remove)}")